In [3]:
##### Assembles rasters on tonnes of livestock products and overall sum of livestock production 
# Distributes FAO country-level production stats to pixels based on livestock density  

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_bounds
import warnings

In [2]:
##### Load data

# Get the current working directory
cd = os.path.dirname(os.getcwd())




In [12]:
##### Define function to get total number of livestock in each pixel 

### INPUTS 

# don't change by animal 
country_total = pd.read_csv(f"{cd}/Data/Clean/Production/FAO_production_by_animal.csv")
country_shp = gpd.read_file("/Users/carinamanitius/Documents/Data/Admin_Boundaries/gadm_410-levels.gpkg", layer='ADM_0')

# change by animal
raster_path = f"{cd}/Data/Raw/FAO_gridded_livestock/GLW4-2020.D-DA.CTL.tif"
output_path = f"{cd}/Data/Clean/Production/Animals/cattle.tif"
column_title = "cattle_t"


### Set-up 

# create dictionary of countries 
country_dict = dict(zip(country_total['ISO3'], country_total['ISO3']))


### LOAD INPUTS

# load raster
with rasterio.open(raster_path) as src:
        density = src.read(1).astype(np.float64)   
        nodata  = src.nodata
        meta    = src.meta.copy()
        transform = src.transform
        crs     = src.crs
        height, width = density.shape

        # Aproximate the area of each pixel in m2 using lat and long of each pixel 
        res_deg = abs(transform.e)           # degrees per pixel (latitude)
        res_lon = abs(transform.a)           # degrees per pixel (longitude)
        # latitude of each row centre
        lats = np.array([
            transform.f + (r + 0.5) * transform.e
            for r in range(height)
        ])
        # area = (lon_res * lat_res) * (111.32 km/deg)² * cos(lat)
        pixel_area_km2 = (
            res_lon * res_deg *
            (111.32 ** 2) *
            np.abs(np.cos(np.radians(lats)))
        )[:, np.newaxis]  # shape (height, 1) – broadcast over columns

if nodata is not None:
    density[density == nodata] = np.nan
density[density < 0] = np.nan


#### CALCULATE
# total animals per pixel = density * area
# density = animial / km2
# area = km2

output = np.full((height, width), np.nan, dtype=np.float64)
animal_per_pixel = density * pixel_area_km2

##### WRITE OUTPUT 

meta.update(
        dtype   = "float32",
        count   = 1,
        nodata  = -9999,
        compress= "lzw"
    )

output = animal_per_pixel

out_arr = output.astype(np.float32)
out_arr[np.isnan(out_arr)] = -9999

with rasterio.open(output_path, "w", **meta) as dst:
    dst.write(out_arr, 1)

total_animals = np.nansum(animal_per_pixel)

print(f"Total animals in raster: {total_animals:.0f}")

Total animals in raster: 1531538615


In [7]:
country_shp

Index(['GID_0', 'COUNTRY', 'geometry'], dtype='object')